# Racing Louisville season comparison

Points per game, expected points per game (xPPG), and final table position for each Racing Louisville regular season, with the number of teams in the league. The current season is included even though it is in progress.

- Standings are computed from regular-season results (ASA `get_games`, `stages="Regular Season"`), ranked by points, then wins, goal difference, goals for.
- xPPG is ASA's `xpoints / count_games` from the regular-season team xgoals endpoint.

Data: American Soccer Analysis

In [1]:
from itscalledsoccer.client import AmericanSoccerAnalysis
import pandas as pd

asa_client = AmericanSoccerAnalysis()

LOU = "eV5DR6YQKn"
SEASONS = [str(s) for s in range(2021, 2027)]
CURRENT_SEASON = SEASONS[-1]


def to_pandas(df):
    # newer itscalledsoccer versions return polars frames
    return df.to_pandas() if hasattr(df, "to_pandas") else df

In [2]:
games_df = to_pandas(
    asa_client.get_games(
        leagues="nwsl", season_name=SEASONS, stages="Regular Season", status="FullTime"
    )
)
games_df = games_df[~games_df.knockout_game]

results_df = pd.concat(
    [
        pd.DataFrame(
            {
                "season_name": games_df.season_name,
                "team_id": games_df[f"{side}_team_id"],
                "goals_for": games_df[f"{side}_score"],
                "goals_against": games_df[f"{other}_score"],
            }
        )
        for side, other in [("home", "away"), ("away", "home")]
    ]
)
results_df["win"] = results_df.goals_for > results_df.goals_against
results_df["points"] = 3 * results_df.win + (
    results_df.goals_for == results_df.goals_against
)

standings_df = (
    results_df.groupby(["season_name", "team_id"])
    .agg(
        games=("win", "size"),
        wins=("win", "sum"),
        points=("points", "sum"),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
    )
    .reset_index()
)
standings_df["goal_difference"] = standings_df.goals_for - standings_df.goals_against
standings_df = standings_df.sort_values(
    ["season_name", "points", "wins", "goal_difference", "goals_for"],
    ascending=[True, False, False, False, False],
)
standings_df["position"] = standings_df.groupby("season_name").cumcount() + 1
standings_df["teams"] = standings_df.groupby("season_name").team_id.transform("size")

In [3]:
xg_df = to_pandas(
    asa_client.get_team_xgoals(
        leagues="nwsl",
        season_name=SEASONS,
        split_by_seasons=True,
        stage_name="Regular Season",
    )
)
xg_df["xppg"] = xg_df.xpoints / xg_df.count_games

lou_df = standings_df.query("team_id == @LOU").merge(
    xg_df[["season_name", "team_id", "xppg"]], on=["season_name", "team_id"]
)
lou_df["ppg"] = lou_df.points / lou_df.games

seasons_table = pd.DataFrame(
    {
        "Season": lou_df.season_name
        + (lou_df.season_name == CURRENT_SEASON).map({True: "*", False: ""}),
        "GP": lou_df.games,
        "PPG": lou_df.ppg.round(2),
        "xPPG": lou_df.xppg.round(2),
        "Position": lou_df.position,
        "Teams": lou_df.teams,
    }
).reset_index(drop=True)
seasons_table

,Season,GP,PPG,xPPG,Position,Teams
0,2021,24,0.92,0.87,9,10
1,2022,22,1.05,1.03,9,12
2,2023,22,1.23,1.21,9,12
3,2024,26,1.08,1.26,9,14
4,2025,26,1.42,1.46,7,14
5,2026*,26,0.88,1.10,15,16


In [4]:
print(seasons_table.to_markdown(index=False))
print(f"\n* {CURRENT_SEASON} in progress: position as of {games_df.date_time_utc.max()}")

| Season   |   GP |   PPG |   xPPG |   Position |   Teams |
|:---------|-----:|------:|-------:|-----------:|--------:|
| 2021     |   24 |  0.92 |   0.87 |          9 |      10 |
| 2022     |   22 |  1.05 |   1.03 |          9 |      12 |
| 2023     |   22 |  1.23 |   1.21 |          9 |      12 |
| 2024     |   26 |  1.08 |   1.26 |          9 |      14 |
| 2025     |   26 |  1.42 |   1.46 |          7 |      14 |
| 2026*    |   26 |  0.88 |   1.1  |         15 |      16 |

* 2026 in progress: position as of 2026-09-26 00:00:00 UTC
